In [2]:
%pip install pandas

  Using cached tzdata-2026.3-py2.py3-none-any.whl.metadata (1.4 kB)
   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   ---- ----------------------------------- 1.0/9.8 MB 5.0 MB/s eta 0:00:02
   ---------------------- ----------------- 5.5/9.8 MB 14.6 MB/s eta 0:00:01
   ---------------------------------------- 9.8/9.8 MB 19.1 MB/s  0:00:00
Using cached tzdata-2026.3-py2.py3-none-any.whl (348 kB)

   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pand

In [6]:
import pandas as pd

df = pd.read_csv("../dataset/Maternal Health Risk Data Set.csv")

df.head()

,Age,SystolicBP,DiastolicBP,BS,BodyTemp,HeartRate,RiskLevel
0,25,130,80,15.0,98.0,86,high risk
1,35,140,90,13.0,98.0,70,high risk
2,29,90,70,8.0,100.0,80,high risk
3,30,140,85,7.0,98.0,70,high risk
4,35,120,60,6.1,98.0,76,low risk


In [7]:
df.shape

(1014, 7)

In [9]:
df.columns

Index(['Age', 'SystolicBP', 'DiastolicBP', 'BS', 'BodyTemp', 'HeartRate',
       'RiskLevel'],
      dtype='str')

In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1014 entries, 0 to 1013
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Age          1014 non-null   int64  
 1   SystolicBP   1014 non-null   int64  
 2   DiastolicBP  1014 non-null   int64  
 3   BS           1014 non-null   float64
 4   BodyTemp     1014 non-null   float64
 5   HeartRate    1014 non-null   int64  
 6   RiskLevel    1014 non-null   str    
dtypes: float64(2), int64(4), str(1)
memory usage: 55.6 KB


In [11]:
df.duplicated().sum()

np.int64(562)

In [13]:
df[df.duplicated(keep=False)].sort_values(
    by=['Age', 'SystolicBP', 'DiastolicBP']
).head(20)

,Age,SystolicBP,DiastolicBP,BS,BodyTemp,HeartRate,RiskLevel
670,10,100,50,6.0,99.0,70,mid risk
849,10,100,50,6.0,99.0,70,mid risk
171,12,90,60,7.9,102.0,66,high risk
267,12,90,60,8.0,102.0,66,high risk
276,12,90,60,11.0,102.0,60,high risk
543,12,90,60,7.5,102.0,66,low risk
552,12,90,60,7.5,102.0,60,low risk
588,12,90,60,7.5,102.0,66,mid risk
827,12,90,60,7.5,102.0,66,mid risk
934,12,90,60,7.5,102.0,66,low risk


In [14]:
duplicate_count = df.duplicated().sum()
duplicate_percentage = (duplicate_count / len(df)) * 100

print("Duplicate rows:", duplicate_count)
print("Duplicate percentage:", duplicate_percentage)

Duplicate rows: 562
Duplicate percentage: 55.42406311637082


In [15]:
duplicate_rows = df[df.duplicated(keep=False)]

duplicate_rows.groupby(
    ['Age', 'SystolicBP', 'DiastolicBP', 'BS', 'BodyTemp', 'HeartRate']
)['RiskLevel'].nunique().value_counts()

RiskLevel
1    261
2     20
3      1
Name: count, dtype: int64

In [16]:
conflicting_rows = (
    df.groupby(
        ['Age', 'SystolicBP', 'DiastolicBP', 'BS', 'BodyTemp', 'HeartRate']
    )['RiskLevel']
    .nunique()
)

print("Conflicting feature combinations:", (conflicting_rows > 1).sum())

Conflicting feature combinations: 35


In [17]:
# Show duplicated rows
duplicates = df[df.duplicated(keep=False)].sort_values(
    by=list(df.columns)
)

duplicates.head(20)

,Age,SystolicBP,DiastolicBP,BS,BodyTemp,HeartRate,RiskLevel
670,10,100,50,6.0,99.0,70,mid risk
849,10,100,50,6.0,99.0,70,mid risk
552,12,90,60,7.5,102.0,60,low risk
940,12,90,60,7.5,102.0,60,low risk
543,12,90,60,7.5,102.0,66,low risk
934,12,90,60,7.5,102.0,66,low risk
588,12,90,60,7.5,102.0,66,mid risk
827,12,90,60,7.5,102.0,66,mid risk
171,12,90,60,7.9,102.0,66,high risk
963,12,90,60,7.9,102.0,66,high risk


In [18]:
# Find feature combinations that have more than one RiskLevel
feature_cols = [
    "Age",
    "SystolicBP",
    "DiastolicBP",
    "BS",
    "BodyTemp",
    "HeartRate"
]

conflicts = (
    df.groupby(feature_cols)["RiskLevel"]
      .nunique()
      .reset_index(name="n_risk_levels")
)

conflicts = conflicts[conflicts["n_risk_levels"] > 1]

conflicts

,Age,SystolicBP,DiastolicBP,BS,BodyTemp,HeartRate,n_risk_levels
4,12,90,60,7.5,102.0,66,2
11,12,95,60,6.9,98.0,65,2
22,13,90,65,7.5,101.0,80,2
53,15,120,80,7.5,98.0,70,2
59,16,100,70,6.9,98.0,80,2
68,17,85,60,9.0,102.0,86,2
79,17,90,65,7.5,103.0,67,2
90,18,90,60,6.9,98.0,70,2
104,19,120,75,6.9,98.0,66,2
109,19,120,80,7.0,98.0,70,2


In [19]:
# Show the actual RiskLevel values for each conflicting combination

conflicts_with_labels = (
    df.groupby(feature_cols)["RiskLevel"]
      .agg(["nunique", lambda x: list(x.unique())])
      .reset_index()
)

conflicts_with_labels.columns = feature_cols + ["n_risk_levels", "risk_levels"]

conflicts_with_labels = conflicts_with_labels[
    conflicts_with_labels["n_risk_levels"] > 1
]

conflicts_with_labels

,Age,SystolicBP,DiastolicBP,BS,BodyTemp,HeartRate,n_risk_levels,risk_levels
4,12,90,60,7.5,102.0,66,2,"[low risk, mid risk]"
11,12,95,60,6.9,98.0,65,2,"[mid risk, low risk]"
22,13,90,65,7.5,101.0,80,2,"[low risk, high risk]"
53,15,120,80,7.5,98.0,70,2,"[mid risk, low risk]"
59,16,100,70,6.9,98.0,80,2,"[mid risk, low risk]"
68,17,85,60,9.0,102.0,86,2,"[mid risk, high risk]"
79,17,90,65,7.5,103.0,67,2,"[low risk, mid risk]"
90,18,90,60,6.9,98.0,70,2,"[mid risk, low risk]"
104,19,120,75,6.9,98.0,66,2,"[mid risk, low risk]"
109,19,120,80,7.0,98.0,70,2,"[mid risk, low risk]"


In [28]:
print("Before removing exact duplicates:", df.shape)

exact_duplicates = df.duplicated().sum()

print("Exact duplicate rows:", exact_duplicates)

Before removing exact duplicates: (1014, 7)
Exact duplicate rows: 562


In [29]:
df_clean = df.drop_duplicates().copy()

print("After removing exact duplicates:", df_clean.shape)

After removing exact duplicates: (452, 7)


In [30]:
print("Remaining exact duplicates:", df_clean.duplicated().sum())

Remaining exact duplicates: 0


In [31]:
df["RiskLevel"].value_counts()

RiskLevel
low risk     406
mid risk     336
high risk    272
Name: count, dtype: int64

In [32]:
df["RiskLevel"].value_counts(normalize=True).round(3)

RiskLevel
low risk     0.400
mid risk     0.331
high risk    0.268
Name: proportion, dtype: float64

In [36]:
df_clean.describe().T

,count,mean,std,min,25%,50%,75%,max
Age,452.0,29.194690,13.767379,10.0,19.0,25.0,35.0,70.0
SystolicBP,452.0,110.553097,17.872282,70.0,90.0,120.0,120.0,160.0
DiastolicBP,452.0,75.418142,13.754578,49.0,65.0,80.0,86.0,100.0
BS,452.0,8.346173,2.829209,6.0,6.9,7.5,7.9,19.0
BodyTemp,452.0,98.692478,1.410897,98.0,98.0,98.0,98.0,103.0
HeartRate,452.0,73.949115,8.156973,7.0,70.0,76.0,80.0,90.0


In [37]:
df_clean.shape

(452, 7)

In [38]:
df_clean.isnull().sum()

Age            0
SystolicBP     0
DiastolicBP    0
BS             0
BodyTemp       0
HeartRate      0
RiskLevel      0
dtype: int64

In [39]:
df_clean.isnull().mean() * 100

Age            0.0
SystolicBP     0.0
DiastolicBP    0.0
BS             0.0
BodyTemp       0.0
HeartRate      0.0
RiskLevel      0.0
dtype: float64

In [43]:
df_clean[df_clean["HeartRate"] < 40]

,Age,SystolicBP,DiastolicBP,BS,BodyTemp,HeartRate,RiskLevel
499,16,120,75,7.9,98.0,7,low risk


In [41]:
df_clean[df_clean["BS"] > 20]

,Age,SystolicBP,DiastolicBP,BS,BodyTemp,HeartRate,RiskLevel


In [42]:
df_clean[df_clean["BodyTemp"] > 104]

,Age,SystolicBP,DiastolicBP,BS,BodyTemp,HeartRate,RiskLevel


In [ ]:
df.loc[499]

In [44]:
df_clean.loc[499]

Age                  16
SystolicBP          120
DiastolicBP          75
BS                  7.9
BodyTemp           98.0
HeartRate             7
RiskLevel      low risk
Name: 499, dtype: object

In [ ]:
df_clean.loc[499, "RiskLevel"]